In [14]:
import pandas as pd 
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

In [15]:
df = pd.read_csv("data/Online Retail.CSV")

In [16]:
print(df.info())
print('\n')
print(df.isnull().sum())
print('\n')
print(df.describe())

#Description
#CustomerID

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB
None


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


            Quantity      UnitPrice     CustomerID
count  541909.000000  541909.000000  406829.000000
mean        9.552250       4.611114   15287.690570
std       218.081158      96.759853    1713.600303
min    -80

In [17]:
df.head(2)
print(df[df['Quantity']<0])

       InvoiceNo StockCode                       Description  Quantity  \
141      C536379         D                          Discount        -1   
154      C536383    35004C   SET OF 3 COLOURED  FLYING DUCKS        -1   
235      C536391     22556    PLASTERS IN TIN CIRCUS PARADE        -12   
236      C536391     21984  PACK OF 12 PINK PAISLEY TISSUES        -24   
237      C536391     21983  PACK OF 12 BLUE PAISLEY TISSUES        -24   
...          ...       ...                               ...       ...   
540449   C581490     23144   ZINC T-LIGHT HOLDER STARS SMALL       -11   
541541   C581499         M                            Manual        -1   
541715   C581568     21258        VICTORIAN SEWING BOX LARGE        -5   
541716   C581569     84978  HANGING HEART JAR T-LIGHT HOLDER        -1   
541717   C581569     20979     36 PENCILS TUBE RED RETROSPOT        -5   

             InvoiceDate  UnitPrice  CustomerID         Country  
141      2010-12-01 9:41      27.50     14527

In [18]:
#return
df_clean=df.copy()
df_clean = df_clean.dropna(subset=['CustomerID', 'Description'])
returns = df_clean[df_clean['InvoiceNo'].astype('str').str.startswith('C')]
df_clean = df_clean[~df_clean['InvoiceNo'].astype('str').str.startswith('C')]
print(len(returns))

8905


In [19]:
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean = df_clean[df_clean['UnitPrice'] > 0]

In [20]:
test_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT']
print(df_clean['StockCode'].isin(test_codes).value_counts())
df_clean = df_clean[~df_clean['StockCode'].isin(test_codes)]


StockCode
False    396470
True       1414
Name: count, dtype: int64


In [21]:
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Revenue'] = df_clean['UnitPrice'] * df_clean['Quantity']

print(df_clean.info())

<class 'pandas.DataFrame'>
Index: 396470 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    396470 non-null  str           
 1   StockCode    396470 non-null  str           
 2   Description  396470 non-null  str           
 3   Quantity     396470 non-null  int64         
 4   InvoiceDate  396470 non-null  datetime64[us]
 5   UnitPrice    396470 non-null  float64       
 6   CustomerID   396470 non-null  int64         
 7   Country      396470 non-null  str           
 8   Revenue      396470 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(4)
memory usage: 30.2 MB
None


In [22]:
returns['CustomerID'] = returns['CustomerID'].astype(int)
returns['InvoiceDate'] = pd.to_datetime(returns['InvoiceDate'])

print(returns.info())

<class 'pandas.DataFrame'>
Index: 8905 entries, 141 to 541717
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   InvoiceNo    8905 non-null   str           
 1   StockCode    8905 non-null   str           
 2   Description  8905 non-null   str           
 3   Quantity     8905 non-null   int64         
 4   InvoiceDate  8905 non-null   datetime64[us]
 5   UnitPrice    8905 non-null   float64       
 6   CustomerID   8905 non-null   int64         
 7   Country      8905 non-null   str           
dtypes: datetime64[us](1), float64(1), int64(2), str(4)
memory usage: 626.1 KB
None


In [23]:
conn = sqlite3.connect("retail.db")

df_clean.to_sql("transactions", conn, if_exists="replace", index=False)

396470

In [24]:
returns.to_sql("returns", conn, if_exists="replace", index=False)

8905

In [25]:
query=""" 
select count(*) as total_rows,
    count(distinct CustomerID) as customers,
    count(distinct StockCode) as products,
    min(InvoiceDate) as date_min,
    max(InvoiceDate) as date_max,
    round(sum(Revenue)) as total_revenue
from transactions;
"""

df=pd.read_sql(query,conn)
df

,total_rows,customers,products,date_min,date_max,total_revenue
0,396470,4334,3660,2010-12-01 08:26:00,2011-12-09 12:50:00,8767753.0
